<a href="https://colab.research.google.com/github/dahlgrenmartin/WaveTracer/blob/main/WaveTracer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install and import required libraries

In [1]:
!pip install -q diffusers transformers accelerate ptwt scikit-learn torchvision

In [2]:
import os
import time
import numpy as np
from sklearn.linear_model import LinearRegression
from PIL import Image

import torch
import torchvision.transforms as T
import ptwt
from huggingface_hub import login
from diffusers import AutoencoderKL

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


# Download SDXL and Real Image
## Create utilities functions

In [3]:
# Skapa en testbild om ingen finns
!wget -q -O real_image.png "https://images.unsplash.com/photo-1544005313-94ddf0286df2?w=1024&q=80" # Standardbild för test
!wget -q -O sdxl_image.png "https://image-b2.civitai.com/file/civitai-media-cache/fe5a6c3a-44cf-48db-82f0-00605e25edf7/original"

def load_vae(model_id, subfolder, device, dtype_str):
    dtype = torch.float16 if dtype_str == "fp16" else torch.float32
    # subfolder="." skapar problem ibland med diffusers, så vi hanterar det
    kwargs = {"torch_dtype": dtype}
    if subfolder and subfolder != ".":
        kwargs["subfolder"] = subfolder

    vae = AutoencoderKL.from_pretrained(model_id, **kwargs)
    vae.to(device)
    vae.eval()
    return vae

def load_image(path, device):
    img = Image.open(path).convert("RGB")
    # VAE förväntar sig normalt värden i intervallet [-1, 1]
    transform = T.Compose([
        T.Resize((1024, 1024)), # SDXL VAE är optimerad för 1024x1024
        T.ToTensor(),
        T.Normalize([0.5], [0.5])
    ])
    return transform(img).unsqueeze(0).to(device)

def encode(vae, target):
    # Skala latents om VAE config kräver det
    posterior = vae.encode(target.to(vae.dtype)).latent_dist
    z = posterior.sample()
    return z * vae.config.scaling_factor

def decode_raw(vae, z):
    z_scaled = z / vae.config.scaling_factor
    return vae.decode(z_scaled).sample

# Choose image to test

In [4]:
INPUT_PATH = "sdxl_image.png"
#INPUT_PATH = "real_image.png"

# RUN THE LATENT REFINEMENT DETECTION
## Instead of tracing slope on FULL PSNR
## Trace it on wavelet subband where effect is biggest


In [5]:
STEPS = 30
LR = 0.01  # 0.01 for SDXL/Wan/SD1.5 and 0.03 for FLUX.2
MODEL_ID = "madebyollin/sdxl-vae-fp16-fix"
VAE_SUBFOLDER = ""

DTYPE = "fp16"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

WAVELET = "sym4"
WINDOW = 10
CHARB_EPS = 2e-3

def charbonnier(x):
    return torch.sqrt(x * x + CHARB_EPS * CHARB_EPS).mean()

def ptwt_input(x):
    n, c, h, w = x.shape
    return x.float().reshape(n * c, 1, h, w)

def slope(values, window):
    y = np.asarray(values[-window:], dtype=np.float64)
    if len(y) < 2:
        return float("nan")
    x = np.arange(len(y)).reshape(-1, 1)
    return float(LinearRegression().fit(x, y).coef_[0])

def psnr(pred, target):
    mse = (pred - target).square().mean().clamp_min(1e-30)
    return float(-10.0 * torch.log10(mse / 4.0))

def main():
    print(f"Laddar VAE till {DEVICE}...")
    vae = load_vae(MODEL_ID, VAE_SUBFOLDER, DEVICE, DTYPE)
    for p in vae.parameters():
        p.requires_grad_(False)

    target = load_image(INPUT_PATH, DEVICE)
    # Cast z to float32 to prevent 'Attempting to unscale FP16 gradients' error
    z = encode(vae, target).detach().clone().to(torch.float32).requires_grad_(True)
    optimizer = torch.optim.Adam([z], lr=LR)
    scaler = torch.amp.GradScaler("cuda", enabled=(DTYPE == "fp16" and DEVICE == "cuda"))

    history = []
    s = float("nan")
    with torch.no_grad():
        # Cast z to VAE's dtype (float16) for decoding to match VAE model's precision
        baseline = psnr(decode_raw(vae, z.to(vae.dtype)), target)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()

    print("Startar optimering...")
    for step in range(1, STEPS + 1):
        # Cast z to VAE's dtype (float16) for decoding in the loop as well
        residual = decode_raw(vae, z.to(vae.dtype)) - target
        coeffs = ptwt.wavedec2(ptwt_input(residual), WAVELET, level=3, mode="zero")

        # level-1 detail bands -> refinement loss
        lh1, hl1, hh1 = coeffs[3]
        loss = torch.abs(lh1).mean() + torch.abs(hl1).mean() + torch.abs(hh1).mean()

        # level-3 off-diagonal residual energy -> detector trace (band-PSNR)
        lh3, hl3, _ = coeffs[1]
        e = 0.5 * (lh3.detach().square().mean() + hl3.detach().square().mean())
        history.append(float(-10.0 * torch.log10(e + 1e-30)))

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        s = slope(history, WINDOW)
        print(f"{step}: L3off_band_psnr={history[-1]:+.3f} slope={s:+.4f}", flush=True)

    if DEVICE == "cuda":
        torch.cuda.synchronize()
    verdict = "SYNTHETIC" if s > 0 else "REAL"
    print(f"Slope={s:+.4f}: {verdict}")
    print(f"wall_clock={time.perf_counter() - t0:.3f}s")

if __name__ == "__main__":
    main()

Laddar VAE till cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Startar optimering...
1: L3off_band_psnr=+24.841 slope=+nan
2: L3off_band_psnr=+24.184 slope=-0.6574
3: L3off_band_psnr=+23.759 slope=-0.5412
4: L3off_band_psnr=+23.599 slope=-0.4152
5: L3off_band_psnr=+23.580 slope=-0.3108
6: L3off_band_psnr=+23.623 slope=-0.2304
7: L3off_band_psnr=+23.679 slope=-0.1710
8: L3off_band_psnr=+23.741 slope=-0.1268
9: L3off_band_psnr=+23.791 slope=-0.0944
10: L3off_band_psnr=+23.836 slope=-0.0703
11: L3off_band_psnr=+23.895 slope=-0.0034
12: L3off_band_psnr=+23.921 slope=+0.0325
13: L3off_band_psnr=+23.952 slope=+0.0451
14: L3off_band_psnr=+23.994 slope=+0.0469
15: L3off_band_psnr=+24.033 slope=+0.0448
16: L3off_band_psnr=+24.083 slope=+0.0429
17: L3off_band_psnr=+24.139 slope=+0.0421
18: L3off_band_psnr=+24.199 slope=+0.0431
19: L3off_band_psnr=+24.252 slope=+0.0449
20: L3off_band_psnr=+24.301 slope=+0.0466
21: L3off_band_psnr=+24.339 slope=+0.0488
22: L3off_band_psnr=+24.369 slope=+0.0489
23: L3off_band_psnr=+24.396 slope=+0.0472
24: L3off_band_psnr=+24.